# Linear Regression on Medical Data for prediction

Goal: Fit and evaluate linear regression on a medical dataset and observe the effect of correlated predictors.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from workshop_utils import load_dataset, basic_train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score



def load_dataset(name: str) -> pd.DataFrame:
    base_url = (
        "https://raw.githubusercontent.com/"
        "Jesse-Islam/QLS-MiCM_Introduction_to_supervised_machine_learning/main/Exercises/data/"
    )
    url = f"{base_url}{name}.csv"
    
    try:
        return pd.read_csv(url)
    except Exception as e:
        raise FileNotFoundError(f"Could not load dataset '{name}' from {url}") from e



def basic_train_test_split(df: pd.DataFrame, target: str, test_size: float = 0.2, random_state: int = 42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def build_preprocessor(num_features, cat_features):
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )
    return preprocessor



In [ ]:


df = load_dataset("bp")
X_train, X_test, y_train, y_test = basic_train_test_split(df, target="sbp")


In [24]:
# TODO 1.1: Fit a simple linear regression using age only
# Tutorial: https://scikit-learn.org/stable/modules/linear_model.html#ordinary-least-squares
model_age = LinearRegression()
model_age.fit(X_train[["age"]], y_train)

# Evaluate predictive performance
y_pred_age = model_age.predict(X_test[["age"]])
print("MAE (age only):", mean_absolute_error(y_test, y_pred_age))
print("R2 (age only):", r2_score(y_test, y_pred_age))

# Inspect coefficient (effect size)
print("Intercept:", model_age.intercept_)
print("Coefficient for age:", model_age.coef_[0])


MAE (age only): 17.725154733899547
R2 (age only): -0.020028365644800594
Intercept: 178.66473539991367
Coefficient for age: 0.1376829391170165


In [ ]:
# ***EXERCISE*** 1.2: Fit a model with multiple predictors
features = ["age", "bmi", "sodium"]
model_multi = LinearRegression()
#fit a linear regression on the data with the specified features

y_pred_multi = #predictions on test set

print("MAE (multi):", mean_absolute_error(y_test, y_pred_multi))
print("R2 (multi):", r2_score(y_test, y_pred_multi))

print("\nCoefficients (effect sizes):")
for f, c in zip(features, model_multi.coef_):
    print(f"{f:10s} {c: .3f}")


MAE (multi): 13.959124302274489
R2 (multi): 0.4118996882898849

Coefficients (effect sizes):
age         0.173
bmi         2.208
sodium      2.900


In [ ]:
# ***EXERCISE*** 1.3: Add correlated feature (weight) and refit
features_corr = []
model_corr = LinearRegression()
model_corr.fit(X_train[features_corr], y_train)

print("Coefficients with correlated predictor:")
for f, c in zip(features_corr, model_corr.coef_):
    print(f"{f:10s} {c: .3f}")


Coefficients with correlated predictor:
age         0.172
bmi         2.521
sodium      2.925
weight     -0.174


In [27]:
# BONUS: Inspect feature correlations
print("\nFeature correlations:")
print(df[features_corr + ["sbp"]].corr(numeric_only=True))



Feature correlations:
             age       bmi    sodium    weight       sbp
age     1.000000 -0.064907  0.060559 -0.043927  0.055665
bmi    -0.064907  1.000000  0.056855  0.826069  0.517824
sodium  0.060559  0.056855  1.000000  0.072072  0.152355
weight -0.043927  0.826069  0.072072  1.000000  0.395635
sbp     0.055665  0.517824  0.152355  0.395635  1.000000


In [28]:
import statsmodels.api as sm
import pandas as pd
X = sm.add_constant(X_train[["age", "bmi", "sodium"]])
ols = sm.OLS(y_train, X).fit()

# Extract key information
results = pd.DataFrame({
    "coef": ols.params,
    "std_err": ols.bse,
    "t_value": ols.tvalues,
    "p_value": ols.pvalues
})

# Display neatly
print(results.round(4))


            coef  std_err  t_value  p_value
const   109.0092  12.2391   8.9066   0.0000
age       0.1734   0.1399   1.2393   0.2171
bmi       2.2083   0.3212   6.8759   0.0000
sodium    2.8997   1.7053   1.7004   0.0910


In [29]:
import statsmodels.api as sm

# Create interaction term manually
df["bmi_sex_interaction"] = df["bmi"] * df["sex"]

X = df[["age", "bmi", "sodium", "exercise_minutes", "sex", "bmi_sex_interaction"]]
y = df["sbp"]

# Fit OLS model
X_ = sm.add_constant(X)
ols = sm.OLS(y, X_).fit()

# Compact output
pd.DataFrame({
    "coef": ols.params,
    "p_value": ols.pvalues
}).round(4)


,coef,p_value
const,87.1910,0.0000
age,0.4836,0.0000
bmi,2.0499,0.0000
sodium,3.6080,0.0000
exercise_minutes,-0.0597,0.0000
sex,6.5811,0.1604
bmi_sex_interaction,0.9335,0.0000


### Reflection:
1. Which predictors have large effect sizes but low p-values (statistically significant)?
2. Which predictors are correlated with each other?
3. Did adding more predictors improve R^2 or MAE meaningfully?
4. If not, what might that tell us about model complexity?
